# 05 - Evaluation & Analysis
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading trained model checkpoints
2. Classification metrics: Accuracy, Precision, Recall, F1, AUC-ROC
3. Confusion matrix visualization
4. ROC and Precision-Recall curves
5. XGBoost feature importance analysis
6. Error analysis (false positives & false negatives)

In [ ]:
import sys
import os
import math
import re
import pickle
import logging
from collections import Counter
from urllib.parse import urlparse

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import DistilBertModel, DistilBertTokenizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report,
)
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
logging.basicConfig(level=logging.WARNING)

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "ml", "checkpoints")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 5.1 Re-define Models & Load Checkpoints

In [ ]:
# ── Helpers ──
def _shannon_entropy(text):
    if not text: return 0.0
    freq = Counter(text); length = len(text)
    return -sum((c/length)*math.log2(c/length) for c in freq.values())

def _has_ip(hostname):
    return 1 if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", hostname) else 0

def _max_cons_consonants(text):
    vowels = set("aeiouAEIOU"); max_c = cur = 0
    for ch in text:
        if ch.isalpha() and ch not in vowels: cur += 1; max_c = max(max_c, cur)
        else: cur = 0
    return max_c

def _vowel_ratio(text):
    alpha = [c for c in text if c.isalpha()]
    if not alpha: return 0.0
    return sum(1 for c in alpha if c in set("aeiouAEIOU")) / len(alpha)

def extract_url_features(url):
    p = urlparse(url); h = p.hostname or ""; path = p.path or ""
    return [len(url), len(h), len(path), url.count("."), url.count("-"), url.count("_"),
            url.count("/"), len(p.query.split("&")) if p.query else 0,
            1 if p.fragment else 0, sum(c.isdigit() for c in url),
            sum(not c.isalnum() and c not in ".-_/:" for c in url),
            _shannon_entropy(url), _shannon_entropy(h), _has_ip(h),
            1 if h.startswith("xn--") else 0,
            1 if p.port and p.port not in (80,443) else 0,
            1 if p.scheme == "https" else 0, 1 if "@" in url else 0,
            1 if "//" in path else 0,
            len(h.split("."))-2 if len(h.split("."))>2 else 0,
            len(h.split(".")[-1]) if "." in h else 0,
            _max_cons_consonants(h), _vowel_ratio(h)]

FEATURE_NAMES = [
    "url_length","hostname_length","path_length","num_dots","num_hyphens",
    "num_underscores","num_slashes","num_query_params","num_fragments",
    "num_digits","num_special_chars","url_entropy","hostname_entropy",
    "has_ip_address","has_punycode","has_port","has_https","has_at_symbol",
    "has_double_slash_redirect","subdomain_count","tld_length",
    "consecutive_consonants_max","vowel_ratio",
]

# ── Model classes ──
class AttentionLayer(nn.Module):
    def __init__(self, h):
        super().__init__(); self.a = nn.Linear(h, 1)
    def forward(self, x):
        w = torch.softmax(self.a(x), dim=1); return torch.sum(w * x, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, out=128, lstm_h=256, layers=2, dropout=0.3, freeze=True):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze:
            for p in self.bert.parameters(): p.requires_grad = False
        bh = self.bert.config.hidden_size
        self.lstm = nn.LSTM(bh, lstm_h, layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers>1 else 0)
        self.attn = AttentionLayer(lstm_h*2)
        self.fc = nn.Linear(lstm_h*2, out)
        self.drop = nn.Dropout(dropout)
    def forward(self, ids, mask):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        x, _ = self.lstm(x); x = self.attn(x)
        return self.fc(self.drop(x))

class MLPBranch(nn.Module):
    def __init__(self, in_dim=23, hidden=[128,64], out=64, drop=0.3):
        super().__init__()
        layers = []; prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]; prev=h
        layers.append(nn.Linear(prev, out))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_feat=23, nlp_out=128, num_out=64, freeze=True):
        super().__init__()
        self.nlp = NLPBranch(out=nlp_out, freeze=freeze)
        self.mlp = MLPBranch(in_dim=num_feat, out=num_out)
        self.fusion_dim = nlp_out + num_out
    def forward(self, ids, mask, feat):
        return torch.cat([self.nlp(ids, mask), self.mlp(feat)], dim=1)

class URLTokenizer:
    def __init__(self, max_len=128):
        self.tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_len = max_len
    def tokenize(self, urls):
        return self.tok(urls, padding=True, truncation=True,
                        max_length=self.max_len, return_tensors="pt")

# ── Load checkpoints ──
fusion_model = PhishScamSenseFusionModel(num_feat=23)
ckpt_path = os.path.join(CHECKPOINT_DIR, "fusion_model.pt")
if os.path.exists(ckpt_path):
    fusion_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Fusion model loaded from checkpoint.")
else:
    print("WARNING: No checkpoint found. Run notebook 04 first.")

xgb_path = os.path.join(CHECKPOINT_DIR, "xgb_classifier.pkl")
if os.path.exists(xgb_path):
    with open(xgb_path, "rb") as f:
        xgb_clf = pickle.load(f)
    print("XGBoost model loaded from checkpoint.")
else:
    print("WARNING: No XGBoost checkpoint found. Run notebook 04 first.")

fusion_model.eval()

## 5.2 Prepare Test Dataset & Get Predictions

In [ ]:
all_urls = [
    "https://www.google.com/search?q=python", "https://github.com/anthropics/claude-code",
    "https://stackoverflow.com/questions/tagged/python", "https://en.wikipedia.org/wiki/Machine_learning",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ", "https://docs.python.org/3/library/urllib.html",
    "https://www.amazon.com/dp/B08N5WRWNW", "https://www.reddit.com/r/MachineLearning",
    "https://mail.google.com/mail/u/0/#inbox", "https://www.linkedin.com/in/johndoe",
    "https://www.microsoft.com/en-us/windows", "https://www.apple.com/macbook-pro",
    "https://www.netflix.com/browse", "https://twitter.com/home", "https://www.bbc.com/news/world",
    "http://192.168.1.1/login/google-verify.html", "http://xn--ggle-1noa.com/accounts/login",
    "http://googl3-security.com/verify?user=admin&token=abc123",
    "http://paypa1-secure.com/signin/update-billing", "http://amaz0n-support.xyz/account/verify",
    "http://microsoft-365-login.tk/auth/signin", "http://netflix-billing-update.ml/payment",
    "http://faceb00k-security.ga/hacked/recovery", "http://apple-id-verify.cf/icloud/login.php",
    "http://bank0famerica-secure.ru/online/login", "http://dhl-tracking-update.info/parcel?id=83927492",
    "http://instagram-verify-account.net/auth", "http://linkedln-security.com/checkpoint/verify",
    "http://dropbox-shared-doc.tk/dl/invoice.pdf.exe", "http://wellsfarg0-alert.com/security/update",
]
all_labels = [0]*15 + [1]*15

tokenizer = URLTokenizer()
tokens = tokenizer.tokenize(all_urls)
num_feat = torch.tensor([extract_url_features(u) for u in all_urls], dtype=torch.float32)

with torch.no_grad():
    fused = fusion_model(tokens["input_ids"], tokens["attention_mask"], num_feat).numpy()

y_proba = xgb_clf.predict_proba(fused)[:, 1]
y_pred  = (y_proba > 0.5).astype(int)
y_true  = np.array(all_labels)

print("Predictions generated.")
print(f"True:  {y_true.tolist()}")
print(f"Pred:  {y_pred.tolist()}")
print(f"Proba: {[f'{p:.2f}' for p in y_proba]}")

## 5.3 Classification Metrics

In [ ]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_proba)

metrics = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "AUC-ROC": auc}

print("=" * 40)
print("  EVALUATION METRICS")
print("=" * 40)
for k, v in metrics.items():
    print(f"  {k:<15} {v:.4f}")
print("=" * 40)
print()
print(classification_report(y_true, y_pred, target_names=["benign", "phishing"]))

## 5.4 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Benign", "Phishing"],
            yticklabels=["Benign", "Phishing"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# Metrics bar chart
axes[1].bar(metrics.keys(), metrics.values(), color=["#3498db","#2ecc71","#e74c3c","#9b59b6","#f39c12"])
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Evaluation Metrics")
axes[1].set_ylabel("Score")
for i, (k, v) in enumerate(metrics.items()):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nTrue Negatives (correct benign):   {tn}")
print(f"False Positives (benign→phishing): {fp}")
print(f"False Negatives (phishing→benign): {fn}")
print(f"True Positives (correct phishing): {tp}")

## 5.5 ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_proba)
axes[0].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()
axes[0].grid(True)

# Precision-Recall Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_proba)
axes[1].plot(rec_curve, prec_curve, color="#3498db", lw=2)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5.6 XGBoost Feature Importance

In [ ]:
# XGBoost feature importance (on the fused 192-dim vector)
# Note: fused dims 0-127 = NLP branch, 128-191 = numerical branch
importance = xgb_clf.feature_importances_
n_features = len(importance)

# Label first 128 as NLP dims, last 64 as numerical feature names
nlp_labels  = [f"nlp_{i}" for i in range(128)]
num_labels  = FEATURE_NAMES   # 23 numerical features

all_labels_importance = nlp_labels + num_labels

# Top 20 most important features
top_k = 20
sorted_idx = np.argsort(importance)[::-1][:top_k]
top_labels = [all_labels_importance[i] for i in sorted_idx]
top_scores = importance[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c" if l.startswith("nlp") else "#3498db" for l in top_labels]
ax.barh(range(top_k), top_scores[::-1], color=colors[::-1])
ax.set_yticks(range(top_k))
ax.set_yticklabels(top_labels[::-1])
ax.set_xlabel("Importance Score")
ax.set_title(f"Top {top_k} XGBoost Feature Importances\n(red=NLP branch, blue=Numerical branch)")
plt.tight_layout()
plt.show()

print(f"\nTotal fused feature dimensions: {n_features}")
print(f"  NLP branch:       128 dims (DistilBERT+BiLSTM+Attention)")
print(f"  Numerical branch:  64 dims (MLP on {len(FEATURE_NAMES)} engineered features)")

## 5.7 Error Analysis: False Positives & False Negatives

In [ ]:
results_df = pd.DataFrame({
    "url": all_urls,
    "true_label": y_true,
    "pred_label": y_pred,
    "confidence": y_proba,
})
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]
results_df["error_type"] = "correct"
results_df.loc[(results_df["true_label"]==0) & (results_df["pred_label"]==1), "error_type"] = "false_positive"
results_df.loc[(results_df["true_label"]==1) & (results_df["pred_label"]==0), "error_type"] = "false_negative"

fp_df = results_df[results_df["error_type"] == "false_positive"]
fn_df = results_df[results_df["error_type"] == "false_negative"]

print(f"False Positives (benign flagged as phishing): {len(fp_df)}")
if not fp_df.empty:
    print(fp_df[["url", "confidence"]].to_string(index=False))

print(f"\nFalse Negatives (phishing missed): {len(fn_df)}")
if not fn_df.empty:
    print(fn_df[["url", "confidence"]].to_string(index=False))

print(f"\nAll results:")
print(results_df[["url", "true_label", "pred_label", "confidence", "error_type"]].to_string(index=False))